# 04 - CNN Model Training

This notebook trains a stronger image-based model for the **Modelling** phase of CRISP-DM.

The previous notebook showed that the HOG + Linear SVM baseline was too weak, especially for the `arch` class. Here, we train a small Convolutional Neural Network (CNN) directly on the rolled fingerprint images.

The task remains:

- `arch`
- `left_slant_loop`
- `right_slant_loop`
- `whorl`

## Why a CNN is the next step

The baseline model used hand-crafted HOG features. That method summarises image edges and ridge directions, but it may miss more complex fingerprint structure.

A CNN learns image features directly from the fingerprint images. This makes it more suitable for recognising ridge-flow patterns such as arches, loops, and whorls.

## Install required libraries

Run this once if the environment does not already have the required packages.

PyTorch is used for the CNN model.

In [ ]:
%pip install pandas numpy matplotlib pillow scikit-learn torch torchvision tqdm

## Section 1: Set up paths and imports

This cell keeps the project paths and imports in one place.

It works whether the notebook is opened from the project root or from inside the `notebooks` folder.

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

current_folder = Path.cwd()
PROJECT_ROOT = current_folder.parent if current_folder.name == "notebooks" else current_folder

PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
MODEL_DIR = PROJECT_ROOT / "models"
FIGURE_DIR = PROCESSED_DIR / "figures"

MODEL_DIR.mkdir(parents=True, exist_ok=True)
FIGURE_DIR.mkdir(parents=True, exist_ok=True)

PROJECT_ROOT

## Section 2: Load the roll-only modelling split

This file was created in the dataset review notebook.

It contains only rolled fingerprint images and uses a subject-aware train, validation, and test split.

In [ ]:
split_path = PROCESSED_DIR / "roll_broad_model_split.csv"

id_columns = {
    "subject_id": "string",
    "finger_position": "string",
    "resolution": "string",
}

dataset = pd.read_csv(split_path, dtype=id_columns)
dataset["image_path"] = dataset["png_path"].apply(lambda path: PROJECT_ROOT / path)

print("Rows:", len(dataset))
print("Subjects:", dataset["subject_id"].nunique())

In [ ]:
pd.crosstab(dataset["broad_class"], dataset["split"])[["train", "validation", "test"]]

## Section 3: Check image files and labels

Before training, we confirm that the image files exist and that class labels are mapped to numeric IDs.

In [ ]:
dataset["image_exists"] = dataset["image_path"].apply(lambda path: path.exists())

if not dataset["image_exists"].all():
    missing_count = (dataset["image_exists"] == False).sum()
    raise FileNotFoundError(f"Missing image files: {missing_count}")

print("All image files were found.")

In [ ]:
label_names = sorted(dataset["broad_class"].unique())
label_to_id = {label: index for index, label in enumerate(label_names)}
id_to_label = {index: label for label, index in label_to_id.items()}

dataset["label_id"] = dataset["broad_class"].map(label_to_id)

label_to_id

## Section 4: Create PyTorch datasets

The training dataset uses light augmentation.

Augmentation helps the model learn more robust patterns by slightly rotating, shifting, and scaling images during training. Validation and test images are not augmented.

In [ ]:
import torch
from torch.utils.data import Dataset, DataLoader
from PIL import Image
from torchvision import transforms

IMAGE_SIZE = 160
BATCH_SIZE = 32

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device

In [ ]:
train_transform = transforms.Compose([
    transforms.Grayscale(num_output_channels=1),
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.RandomRotation(8),
    transforms.RandomAffine(degrees=0, translate=(0.03, 0.03), scale=(0.95, 1.05)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.5], std=[0.5]),
])

eval_transform = transforms.Compose([
    transforms.Grayscale(num_output_channels=1),
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.5], std=[0.5]),
])

In [ ]:
class FingerprintDataset(Dataset):
    def __init__(self, frame, transform):
        self.frame = frame.reset_index(drop=True)
        self.transform = transform

    def __len__(self):
        return len(self.frame)

    def __getitem__(self, index):
        row = self.frame.iloc[index]
        image = Image.open(row["image_path"])
        image = self.transform(image)
        label = int(row["label_id"])
        return image, label

In [ ]:
train_data = dataset[dataset["split"] == "train"].copy()
validation_data = dataset[dataset["split"] == "validation"].copy()
test_data = dataset[dataset["split"] == "test"].copy()

train_dataset = FingerprintDataset(train_data, train_transform)
validation_dataset = FingerprintDataset(validation_data, eval_transform)
test_dataset = FingerprintDataset(test_data, eval_transform)

len(train_dataset), len(validation_dataset), len(test_dataset)

In [ ]:
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=0)
validation_loader = DataLoader(validation_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)

## Section 5: Preview training images

This verifies that the CNN receives clean grayscale fingerprint tensors.

In [ ]:
images, labels = next(iter(train_loader))

print("Image batch:", images.shape)
print("Label batch:", labels.shape)

In [ ]:
fig, axes = plt.subplots(3, 4, figsize=(10, 7))

for axis, image, label in zip(axes.ravel(), images[:12], labels[:12]):
    axis.imshow(image.squeeze(), cmap="gray")
    axis.set_title(id_to_label[int(label)])
    axis.axis("off")

plt.tight_layout()
plt.savefig(FIGURE_DIR / "cnn_training_batch_preview.png", dpi=150)
plt.show()

## Stop and review CNN data setup

Pause here before training.

Check that:

- all image files were found.
- the train, validation, and test sizes are correct.
- the image batch shape looks like `(batch, 1, 160, 160)`.
- the preview image grid shows fingerprint images with labels.

After this, continue to model training.

## Section 6: Define the CNN model

This is a small CNN built for a first deep learning experiment.

It learns ridge-flow features from images using convolution layers, then predicts one of the four broad pattern classes.

In [ ]:
import torch.nn as nn

class FingerprintCNN(nn.Module):
    def __init__(self, number_of_classes):
        super().__init__()

        self.features = nn.Sequential(
            nn.Conv2d(1, 16, kernel_size=3, padding=1),
            nn.BatchNorm2d(16),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(16, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.MaxPool2d(2),
        )

        self.classifier = nn.Sequential(
            nn.AdaptiveAvgPool2d(1),
            nn.Flatten(),
            nn.Dropout(0.30),
            nn.Linear(64, number_of_classes),
        )

    def forward(self, images):
        features = self.features(images)
        return self.classifier(features)

In [ ]:
model = FingerprintCNN(number_of_classes=len(label_names)).to(device)

model

## Section 7: Handle class imbalance

The `arch` class has fewer examples than the loop and whorl classes.

Class weights tell the loss function to penalise mistakes on smaller classes more strongly.

In [ ]:
train_class_counts = train_data["broad_class"].value_counts().reindex(label_names)
class_weights = len(train_data) / (len(label_names) * train_class_counts)

class_weights = torch.tensor(class_weights.values, dtype=torch.float32).to(device)

train_class_counts

## Section 8: Train the CNN

We train for a small number of epochs first.

The best model is selected using validation accuracy, not test accuracy.

In [ ]:
import torch.optim as optim

criterion = nn.CrossEntropyLoss(weight=class_weights)
optimizer = optim.Adam(model.parameters(), lr=0.001)

EPOCHS = 12

In [ ]:
def run_epoch(model, data_loader, criterion, optimizer=None):
    training = optimizer is not None
    model.train() if training else model.eval()

    total_loss = 0
    total_correct = 0
    total_images = 0

    for images, labels in data_loader:
        images = images.to(device)
        labels = labels.to(device)

        with torch.set_grad_enabled(training):
            outputs = model(images)
            loss = criterion(outputs, labels)

            if training:
                optimizer.zero_grad()
                loss.backward()
                optimizer.step()

        total_loss += loss.item() * images.size(0)
        total_correct += (outputs.argmax(1) == labels).sum().item()
        total_images += images.size(0)

    return total_loss / total_images, total_correct / total_images

In [ ]:
history = []
best_validation_accuracy = 0
best_model_path = MODEL_DIR / "cnn_roll_baseline.pt"

for epoch in range(1, EPOCHS + 1):
    train_loss, train_accuracy = run_epoch(model, train_loader, criterion, optimizer)
    validation_loss, validation_accuracy = run_epoch(model, validation_loader, criterion)

    history.append({
        "epoch": epoch,
        "train_loss": train_loss,
        "train_accuracy": train_accuracy,
        "validation_loss": validation_loss,
        "validation_accuracy": validation_accuracy,
    })

    if validation_accuracy > best_validation_accuracy:
        best_validation_accuracy = validation_accuracy
        torch.save(model.state_dict(), best_model_path)

    print(f"Epoch {epoch:02d}: train_acc={train_accuracy:.3f}, val_acc={validation_accuracy:.3f}")

print("Best validation accuracy:", round(best_validation_accuracy, 4))

## Section 9: Plot training history

These plots show whether the model is learning and whether training and validation performance move together.

In [ ]:
history_table = pd.DataFrame(history)
history_path = PROCESSED_DIR / "cnn_training_history.csv"

history_table.to_csv(history_path, index=False)
history_table

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))

axes[0].plot(history_table["epoch"], history_table["train_accuracy"], label="train")
axes[0].plot(history_table["epoch"], history_table["validation_accuracy"], label="validation")
axes[0].set_title("CNN Accuracy")
axes[0].set_xlabel("Epoch")
axes[0].set_ylabel("Accuracy")
axes[0].legend()

axes[1].plot(history_table["epoch"], history_table["train_loss"], label="train")
axes[1].plot(history_table["epoch"], history_table["validation_loss"], label="validation")
axes[1].set_title("CNN Loss")
axes[1].set_xlabel("Epoch")
axes[1].set_ylabel("Loss")
axes[1].legend()

plt.tight_layout()
plt.savefig(FIGURE_DIR / "cnn_training_history.png", dpi=150)
plt.show()

## Stop and review CNN training

Pause here after training.

Check the validation accuracy and the training-history plot before running the final test evaluation.

## Section 10: Evaluate the best CNN on the test set

This section can run on its own after `cnn_roll_baseline.pt` has been saved.

The test set is used for the final evaluation.

In [ ]:
model = FingerprintCNN(number_of_classes=len(label_names)).to(device)
model.load_state_dict(torch.load(MODEL_DIR / "cnn_roll_baseline.pt", map_location=device))
model.eval()

print("Loaded best CNN model.")

In [ ]:
def collect_predictions(model, data_loader):
    all_predictions = []
    all_labels = []

    model.eval()
    with torch.no_grad():
        for images, labels in data_loader:
            images = images.to(device)
            outputs = model(images)

            all_predictions.extend(outputs.argmax(1).cpu().numpy())
            all_labels.extend(labels.numpy())

    return np.array(all_labels), np.array(all_predictions)

In [ ]:
from sklearn.metrics import accuracy_score, classification_report

test_labels, test_predictions = collect_predictions(model, test_loader)

test_accuracy = accuracy_score(test_labels, test_predictions)
print("CNN test accuracy:", round(test_accuracy, 4))

In [ ]:
cnn_report = classification_report(
    test_labels,
    test_predictions,
    target_names=label_names,
    output_dict=True,
)

cnn_report_table = pd.DataFrame(cnn_report).T
cnn_report_table.round(3)

## Section 11: CNN confusion matrix

This shows which classes the CNN still confuses after learning directly from images.

In [ ]:
from sklearn.metrics import ConfusionMatrixDisplay

fig, axis = plt.subplots(figsize=(7, 6))

ConfusionMatrixDisplay.from_predictions(
    test_labels,
    test_predictions,
    display_labels=label_names,
    xticks_rotation=25,
    cmap="Blues",
    ax=axis,
)

axis.set_title("CNN Confusion Matrix")
plt.tight_layout()
plt.savefig(FIGURE_DIR / "cnn_confusion_matrix.png", dpi=150)
plt.show()

## Section 12: Save CNN evaluation outputs

The report and prediction table are saved for the final project write-up.

In [ ]:
cnn_report_path = PROCESSED_DIR / "cnn_test_classification_report.csv"
cnn_report_table.to_csv(cnn_report_path)

test_results = test_data.copy().reset_index(drop=True)
test_results["true_label"] = [id_to_label[index] for index in test_labels]
test_results["predicted_label"] = [id_to_label[index] for index in test_predictions]
test_results["correct"] = test_results["true_label"] == test_results["predicted_label"]

prediction_path = PROCESSED_DIR / "cnn_test_predictions.csv"
test_results.to_csv(prediction_path, index=False)

print("Saved:", cnn_report_path)
print("Saved:", prediction_path)

## Stop and review the CNN model

Pause here after test evaluation.

We should compare the CNN against the HOG baseline using:

- test accuracy
- macro F1-score
- per-class recall
- confusion matrix patterns

If the CNN improves the baseline, it becomes the main model. If not, the next step is transfer learning or better preprocessing.